<a href="https://colab.research.google.com/github/uddipta-deka/multiclass-brain-tumor-detection/blob/main/notebooks/3_Transfer_learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Transfer Learning Model Comparison for Brain Tumor Classification

In this notebook, I am comparing multiple pretrained models (VGG16, ResNet50, EfficientNetB0) with the baseline CNN model to evaluate performance improvements.

In [9]:
import os

BASE_PATH = "/content/drive/MyDrive/Brain Tumor Classification"

TRAIN_DIR = os.path.join(BASE_PATH, "dataset", "Training")
TEST_DIR = os.path.join(BASE_PATH, "dataset", "Testing")


In [4]:
import os
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')


import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator


from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve


In [5]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

In [10]:
#Training Augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest',
    validation_split=0.2
)


test_datagen = ImageDataGenerator(rescale=1./255)


train_gen = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True
)

val_gen = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=True
)

test_gen = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

Found 4480 images belonging to 4 classes.
Found 1120 images belonging to 4 classes.
Found 1600 images belonging to 4 classes.


In [11]:
print("Class Indices:", train_gen.class_indices)

Class Indices: {'glioma': 0, 'meningioma': 1, 'notumor': 2, 'pituitary': 3}


**Experiment 1: VGG16 (Classic Transfer Learning Approach)**

VGG16 is a well-known deep convolutional network. I am using it as the first transfer learning model to see how a standard architecture performs on this dataset.

Strategy: Two-Stage Training

**Stage 1 (Feature Extraction):**

Freezing the VGG16 base model and train only the custom classification layers.

**Stage 2 (Fine-Tuning):**

Unfreezing the top layers of the model and train with a low learning rate to adapt the features to MRI images.

In [12]:
from tensorflow.keras.applications import VGG16
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

# VGG16 base
base_vgg = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))


base_vgg.trainable = False

# Classification head
x = base_vgg.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation='relu')(x)
x = BatchNormalization()(x)
x = Dropout(0.5)(x)
predictions = Dense(4, activation='softmax')(x)

#  Final model
model_vgg = Model(inputs=base_vgg.input, outputs=predictions)


model_vgg.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model_vgg.summary()

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv1 (Conv2D)           │ (None, 224, 224, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv2 (Conv2D)           │ (None, 224, 224, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_pool (MaxPooling2D)      │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv1 (Conv2D)           │ (None, 112, 112, 128)  │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv2 (Conv2D)           │ (None, 112, 112, 128)  │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_pool (MaxPooling2D)      │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv1 (Conv2D)           │ (None, 56, 56, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv2 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv3 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_pool (MaxPooling2D)      │ (None, 28, 28, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv1 (Conv2D)           │ (None, 28, 28, 512)    │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv2 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv3 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_pool (MaxPooling2D)      │ (None, 14, 14, 512)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv1 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv2 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv3 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_pool (MaxPooling2D)      │ (None, 7, 7, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 512)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │         1,02

 Total params: 14,848,068 (56.64 MB)

 Trainable params: 132,868 (519.02 KB)

 Non-trainable params: 14,715,200 (56.13 MB)

**Model Design Choices**

GlobalAveragePooling2D: Used to reduce parameters and avoid overfitting compared to Flatten.

BatchNormalization: Helps stabilize training of the new layers.

Dropout (0.5): Added to reduce overfitting.

Learning Rate (1e-3): Used for faster training in Stage 1 since the base model is frozen.

In [14]:
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

# Define Callbacks for Stage 1
SAVE_PATH = os.path.join(BASE_PATH, 'models')
os.makedirs(SAVE_PATH, exist_ok=True)

checkpoint_vgg_s1 = ModelCheckpoint(
    filepath=os.path.join(SAVE_PATH, 'vgg16_stage1.keras'),
    monitor='val_loss',
    save_best_only=True,
    verbose=1
)

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)


reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.2,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

# TRAINING
print("Training VGG16 Stage 1:")
history_vgg_s1 = model_vgg.fit(
    train_gen,
    epochs=15,
    validation_data=val_gen,
    callbacks=[checkpoint_vgg_s1, early_stop, reduce_lr]
)

Training VGG16 Stage 1:
Epoch 1/15
  7/140 ━━━━━━━━━━━━━━━━━━━━ 40:39 18s/step - accuracy: 0.3034 - loss: 1.6111

KeyboardInterrupt: 

### VGG16 Stage 1: Results & Observations

**The "Stage-1" phase has successfully concluded with the following metrics:**

**Best Validation Accuracy:** 91.61%

**Final Validation Loss:** 0.2559

**Learning Rate Adjustments:** The model required one decay (Epoch 6) to stabilize.

**Analysis**

VGG16 Stage 1 has already outperformed the custom CNN baseline (86%) by 5.61%. The high validation accuracy relative to training accuracy (~90%) indicates that the Dropout(0.5) and BatchNormalization layers are effectively regularizing the model, preventing overfitting even with a large pre-trained base.

## Stage 2: Fine-Tuning using Functional Blocks
In this stage, I am performing fine-tuning of the pretrained model to further improve performance. Instead of unfreezing the entire network or using a fixed percentage of layers, I follow a functional block-based approach.

**Strategy :**

Only the final convolutional block of the pretrained model is unfrozen.

Earlier layers remain frozen to preserve low-level features such as edges and textures.

The unfrozen layers are trained with a very low learning rate (1e-5) to make small, controlled updates.

**Rationale :**

Different architectures have different depths and internal structures, so unfreezing a fixed number or percentage of layers may not be consistent across models. By targeting the last functional block, I ensure that:


High-level, task-specific features are adapted to the MRI dataset

Pretrained low-level representations are preserved

The comparison between models remains fair and meaningful

**Model-Specific Fine-Tuning**

VGG16: Unfreeze Block 5 (final convolutional block)

ResNet50: Unfreeze Stage 5 (final residual block)

EfficientNetB0: Unfreeze Block 7 (final MBConv block)

In [15]:
from tensorflow.keras.models import load_model


# Stage 1 model
SAVE_PATH = os.path.join(BASE_PATH, 'models')
model_vgg = load_model(os.path.join(SAVE_PATH, 'vgg16_stage1.keras'))

# Unfreezinng only the Final Functional Block (Block 5)

for layer in model_vgg.layers:

    if 'block5' in layer.name:
        layer.trainable = True
    # everything else (Block 1-4 and the Input) frozen


    elif 'block' in layer.name or 'input' in layer.name:
        layer.trainable = False


model_vgg.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Checkpoint for Stage 2
checkpoint_vgg_s2 = ModelCheckpoint(
    filepath=os.path.join(SAVE_PATH, 'vgg16_finetuned.keras'),
    monitor='val_loss',
    save_best_only=True,
    verbose=1
)


for layer in model_vgg.layers:
    print(f"Layer: {layer.name} | Trainable: {layer.trainable}")

#  Fine-Tuning
print("Fine-tuning Block 5...")
history_vgg_s2 = model_vgg.fit(
    train_gen,
    epochs=10,
    validation_data=val_gen,
    callbacks=[checkpoint_vgg_s2, early_stop, reduce_lr]
)

Layer: input_layer | Trainable: False
Layer: block1_conv1 | Trainable: False
Layer: block1_conv2 | Trainable: False
Layer: block1_pool | Trainable: False
Layer: block2_conv1 | Trainable: False
Layer: block2_conv2 | Trainable: False
Layer: block2_pool | Trainable: False
Layer: block3_conv1 | Trainable: False
Layer: block3_conv2 | Trainable: False
Layer: block3_conv3 | Trainable: False
Layer: block3_pool | Trainable: False
Layer: block4_conv1 | Trainable: False
Layer: block4_conv2 | Trainable: False
Layer: block4_conv3 | Trainable: False
Layer: block4_pool | Trainable: False
Layer: block5_conv1 | Trainable: True
Layer: block5_conv2 | Trainable: True
Layer: block5_conv3 | Trainable: True
Layer: block5_pool | Trainable: True
Layer: global_average_pooling2d | Trainable: True
Layer: dense | Trainable: True
Layer: batch_normalization | Trainable: True
Layer: dropout | Trainable: True
Layer: dense_1 | Trainable: True
Fine-tuning Block 5...
Epoch 1/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 0s 22s/step - 

### VGG16 Stage 2: Fine-Tuning Results & Observations

**The fine-tuning phase has significantly improved the model’s performance:**

Best Validation Accuracy: 96.07%

Best Validation Loss : 0.1289

Learning Rate Adjustments: One decay applied (Epoch 9) for stabilization.

**Analysis:**

Fine-tuning has led to a major performance boost compared to Stage 1. By allowing the final convolutional block to adapt to MRI-specific features, the model achieved better generalization and sharper class discrimination.

Training accuracy increased to ~97%, indicating strong learning capacity

Validation accuracy improved without instability, showing controlled fine-tuning

Lower validation loss confirms improved confidence in predictions


**Comparison with Stage 1**

| Metric               | Stage 1 | Stage 2 (Fine-Tuned) |
|---------------------|--------|----------------------|
| Validation Accuracy | **91.61%** | **96.07%**              |
| Validation Loss     | **0.2559** | **0.1289**              |

**Improvement :**		+4.46% accuracy gain

**Key Observations**

Significant Improvement: Stage 2 improved accuracy by ~4.46%, which is substantial for fine-tuning

Better Feature Adaptation: Unfreezing Block 5 allowed the model to learn high-level tumor-specific patterns

Stable Training: Despite deeper training, no major overfitting was observed

Effective Strategy: Functional block fine-tuning proved to be an efficient and stable approach for adapting the pretrained model to the MRI dataset.